# 0.4 概率论、随机性与模拟：量化交易的“排雷演习”

> **“如果历史再来一次，我的策略还会赚钱吗？”**
> 
> 量化交易本质上是在**不确定性**中寻找**概率优势**。回测只是“这辈子”发生过的事，但金融市场有无数种可能的变数。
> 
> **这一节我们将深入普及量化金融中最核心的统计工具，教你如何用科学的眼光看待风险。**

## 学习目标
- **概率分布全家桶**：正态、对数正态、学生 t 分布（捕获黑天鹅）。
- **统计基石**：直观演示大数定律 (LLN) 与中心极限定理 (CLT)。
- **模拟进阶**：从蒙特卡洛 (GBM) 到数据重采样 (Bootstrapping)。
- **风险管理**：实战计算 Value at Risk (VaR) 与 CVaR。
- **思维升维**：初步理解贝叶斯思维（根据新数据更新逻辑）。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf
import seaborn as sns

plt.style.use('seaborn-v0_8-muted')
plt.rcParams['figure.figsize'] = (10, 6)
print("环境准备就绪 ✅")

## 1. 概率分布：金融世界的真实形状

### 1.1 正态分布 vs. 对数正态分布
- **正态分布 (Normal)**：假设收益率分布在左右两边是对称的。但有个致命问题：它允许价格变成负数（这不符合现实）。
- **对数正态 (Log-Normal)**：金融学假设**价格**服从对数正态分布。它保证了价格永远大于 0，且允许右侧有长长的“暴富”尾巴。

### 1.2 学生 t 分布：捕获“黑天鹅”
正态分布预测的暴跌频率极低。但在真实市场，暴跌发生的次数多得多。**学生 t 分布**比正态分布有更“厚”的尾巴，更能描述市场的真实风险。

In [ ]:
# 理论分布对比
x = np.linspace(-4, 4, 1000)
norm_pdf = stats.norm.pdf(x, 0, 1)
t_pdf = stats.t.pdf(x, df=3) # 自由度低 = 尾部更厚

plt.plot(x, norm_pdf, label='正态分布 (理论)', lw=2)
plt.plot(x, t_pdf, label='学生 t 分布 (厚尾)', lw=2, linestyle='--')
plt.fill_between(x, 0, t_pdf, where=(x < -2.5), color='red', alpha=0.3, label='潜在黑天鹅区域')
plt.title("为什么我们需要厚尾分布？")
plt.legend()
plt.show()

## 2. 统计基石：为什么回测是有效的？

### 2.1 大数定律 (LLN)：勤能补拙
如果你有一个胜率 51% 的策略，做 10 次交易可能还亏钱。但只要你交易 10,000 次，你的平均收益率就一定会回归到那 1% 的优势上。**这就是大数定律：成交次数越多，偶然性越低。**

### 2.2 中心极限定理 (CLT)：噪声的终结
这就是为什么不管原始数据多乱，最后计算出的“平均值”往往呈正态分布。它给了我们用简单统计模型分析复杂市场的底气。

In [ ]:
# 演示 CLT：从随机乱序到正态分布
np.random.seed(42)
n_samples = 1000
sample_size = 50

# 随机生成 10000 个乱七八糟的数字（均匀分布）
population = np.random.uniform(0, 1, 10000)

# 抽取 1000 组样本，计算每组的平均值
means = [np.random.choice(population, sample_size).mean() for _ in range(n_samples)]

plt.hist(means, bins=30, density=True, color='skyblue', alpha=0.7)
plt.title(f"中心极限定理演示：{sample_size} 个随机变量的均值分布（已呈正态！）")
plt.show()

## 3. 蒙特卡洛模拟：预演 100 种未来

### 3.1 基础：几何布朗运动 (GBM)
这是金融建模中最经典的随机过程。假设价格由“趋势（漂移）”和“波动（扩散）”两部分组成。

In [ ]:
def gbm_simulation(s0, mu, sigma, t, n_paths):
    dt = 1/252
    n_steps = int(t * 252)
    paths = np.zeros((n_steps + 1, n_paths))
    paths[0] = s0
    for i in range(1, n_steps + 1):
        # dS = mu*S*dt + sigma*S*dW
        shock = np.random.normal(0, np.sqrt(dt), n_paths)
        paths[i] = paths[i-1] * (1 + mu * dt + sigma * shock)
    return paths

# 模拟 SPY 未来一年的 50 种走势
paths = gbm_simulation(s0=500, mu=0.1, sigma=0.15, t=1, n_paths=50)
plt.plot(paths, color='gray', alpha=0.2)
plt.plot(paths.mean(axis=1), color='red', lw=2, label='平均预期路径')
plt.title("蒙特卡洛模拟 (GBM)：未来一年的各种可能")
plt.legend()
plt.show()

### 3.2 进阶：Bootstrapping (自助法重采样)
**与其假设价格服从某个公式，不如直接从历史数据里“切片”拼凑。** 
比如我们随机抽取历史上真实的 10 天收益率，拼成一个模拟的未来。这种方法不需要假设分布，更能反映真实市场的“怪癖”。

In [ ]:
spy = yf.download('SPY', start='2010-01-01', end='2024-01-01', progress=False)['Close'].squeeze()
spy_rets = spy.pct_change().dropna()

# Bootstrapping: 从历史收益率中随机有放回地抽取
n_days = 252
n_sims = 1000
sim_rets = np.random.choice(spy_rets, (n_days, n_sims))
sim_cumulative = (1 + sim_rets).cumprod(axis=0) * spy.iloc[-1]

plt.plot(sim_cumulative[:, :50], color='blue', alpha=0.1)
plt.title("Bootstrapping 模拟：利用真实历史收益率随机拼凑未来")
plt.show()

## 4. 风险管理：我今天可能会亏多少？

### 4.1 Value at Risk (VaR) — 在位风险
“我有 95% 的信心，明天亏损不会超过 X 元。”——这里的 X 就是 VaR。

### 4.2 CVaR (Conditional VaR) — 预期亏损
VaR 只告诉你“门槛”。如果那 5% 的极端情况真的发生了，平均会亏多少？这就是 CVaR。它比 VaR 更稳健，因为它考虑了那部分“最惨烈”的情况。

In [ ]:
# 利用刚才 Bootstrapping 模拟的结果计算风险
final_period_rets = sim_cumulative[-1] / spy.iloc[-1] - 1
alpha = 0.05 # 95% 置信水平

var_95 = np.percentile(final_period_rets, alpha * 100)
cvar_95 = final_period_rets[final_period_rets <= var_95].mean()

sns.histplot(final_period_rets, bins=50, kde=True)
plt.axvline(var_95, color='red', linestyle='--', label=f'VaR (95%): {var_95:.2%}')
plt.axvline(cvar_95, color='darkred', lw=2, label=f'CVaR (95%): {cvar_95:.2%}')
plt.title("利用模拟结果量化一年后的持仓风险")
plt.legend()
plt.show()

print(f"结论：如果你现在买入并持有，一年后有 5% 的概率亏损超过 {abs(var_95):.2%}")
print(f"一旦进入最坏的那 5%，平均亏损将达到 {abs(cvar_95):.2%}")

## 5. 贝叶斯思维：动态更新你的世界观

> **“贝叶斯定理是量化交易的灵魂。”**
>
> 它告诉我们：**你的新观点 = 旧经验（先验） + 新观测的数据（似然）。**

**例子**：
- **旧经验**：你认为这只股票长期是涨的。
- **新数据**：最近财报暴雷，股价连续跌停。
- **贝叶斯更新**：你不能死抱着“长期看涨”不放，你需要根据新跌势，迅速下调对它的预期胜率。

在后续的高级策略中，我们会学到如何用卡尔曼滤波（Kalman Filter）等工具实现这种“动态演进”。

## 🎯 总结与思考

1. **别被正态分布骗了**：市场总是在极端情况下“杀人”，学会用厚尾分布思考。
2. **靠模拟演习**：在实盘放钱前，先用蒙特卡洛或 Bootstrapping 跑 10,000 遍。
3. **守住红线**：始终知道你的 **VaR** 在哪里，别让自己在一次意外中出局。

---
**下一模块** → `02_data/`（我们将学习如何严谨地获取和清洗数据，为这些复杂的分析打好地基）